In [1]:
import pandas as pd
import sys
import numpy as np
import warnings
import os

sys.path.append("/Users/ejowik001/Desktop/Github/Nowcasting/kedro/refinery/dependencies/")

In [2]:
from estimation import cast_to_base_unit, calculate_contributions
from plots import plot_prediction
from retransform_prediction import retransform_
from retransform_data import retransform_data
from utils import cast_spec_to_dict, _convert_to_datetime

In [3]:
from dateutil.relativedelta import relativedelta

In [4]:
# !pip install shap
# import shap

In [5]:
def calculate_conf_bounds(pred, actual):
    # mape_series = np.abs((actual - pred) / actual).expanding(1).mean()
    # upper = (1+mape_series)*pred
    # lower = (1-mape_series)*pred
    std = (actual - pred).shift(1).expanding(2).std().fillna(0)
    low1, upp1 = pred-std, pred+std
    low2, upp2 = pred-3*std, pred+3*std
    return {"L1": low1, "U1": upp1, "L2": low2, "U2": upp2}

In [6]:
EPSILON = 1e-10

def rrse(actual: np.ndarray, predicted: np.ndarray, benchmark: np.ndarray=None):
    """ Root Relative Squared Error """
    return np.sqrt(
        np.sum(np.square(actual - predicted))
        / np.sum(np.square(actual - benchmark))
    )

def _error(actual: np.ndarray, predicted: np.ndarray):
    """ Simple error """
    return actual - predicted

def _percentage_error(actual: np.ndarray, predicted: np.ndarray):
    """
    Percentage error

    Note: result is NOT multiplied by 100
    """
    return _error(actual, predicted) / (actual + EPSILON)

def mape(actual: np.ndarray, predicted: np.ndarray):
    """
    Mean Absolute Percentage Error

    Note: result is NOT multiplied by 100
    """
    return np.mean(np.abs(_percentage_error(actual, predicted)))


def mse(actual: np.ndarray, predicted: np.ndarray):
    """ Mean Squared Error """
    return np.mean(np.square(_error(actual, predicted)))


def rmse(actual: np.ndarray, predicted: np.ndarray):
    """ Root Mean Squared Error """
    return np.sqrt(mse(actual, predicted))


In [7]:
# def assign_weights(s):
#     if (s['directional_accuracy'] == -1) and (s['within_cbounds'] == -1):
#         return 2
#     elif (s['directional_accuracy'] == -1) and (s['within_cbounds'] == 1):
#         return 1.75
#     elif (s['directional_accuracy'] == 1) and (s['within_cbounds'] == -1):
#         return 1.25
#     elif (s['directional_accuracy'] == 1) and (s['within_cbounds'] == 1):
#         return 1
#     else: return np.infty

# confidence_bounds_func = lambda row: row['lower']<=row['y_pred']<=row['upper']

# def wdmpe(predicted, actual):
#     actual_diff = actual.sort_index().diff()
#     actual_signs = np.sign(actual_diff)
#     predicted_diff = predicted.sort_index().diff()
#     predicted_signs = np.sign(predicted_diff)

#     resid = predicted-actual

#     dir_acc = list(actual_signs * predicted_signs)

#     resid_mean = resid.expanding(1).mean()
#     resid_std = resid.expanding(2).std().fillna(0)

#     lower = actual-resid_std
#     upper = actual+resid_std

#     df = pd.DataFrame({
#         "directional_accuracy": dir_acc,
#         "lower": lower,
#         "upper": upper,
#         "y_pred": predicted
#     }).iloc[1:, :]
#     df['within_cbounds'] = df.apply(confidence_bounds_func, axis=1).astype(int).replace({0: -1})
#     df['percentage_error'] = resid / actual

#     df['weights'] = df.apply(assign_weights, axis=1)
#     df["weighted_percentage_error"] = df['weights'] * df['percentage_error']
#     return df["weighted_percentage_error"].mean()


In [8]:
# def cast_to_base_unit(ds, model_result, spec, series_name):
#     Spec = cast_spec_to_dict(spec.loc[spec["seriesid"] == series_name])

#     ## Retransform
#     ds = _convert_to_datetime(ds, ['ReferenceDate'])

#     dsrc = ds.set_index('ReferenceDate')

#     # def retransform_prediction(transf_series, base_series, Spec, series_name):
#     base_series = dsrc[series_name]
#     header = [series_name]

#     backcast = model_result['predictions']['backcast']
#     forecast = pd.Series(
#         model_result["predictions"]["forecast"],
#         index=[model_result["predictions"]["reference_date"]]
#         )

#     transf_pred = pd.concat([backcast, forecast])
#     transf_pred.index = pd.to_datetime(transf_pred.index)

#     transf_series = model_result["actual"]

#     Time = np.sort(np.unique(np.concatenate((base_series.index.date, transf_pred.index.date))))
#     cutoff_date = transf_pred.index.min().date()

#     Z = base_series.reindex(Time).to_numpy().reshape(-1,1)

#     Yhat = transf_pred.reindex(Time).to_numpy().reshape(-1,1)
#     Y = transf_series.reindex(Time).to_numpy().reshape(-1,1)

#     Rhat = retransform_(X=Yhat, Z=Z, Time=Time, Spec=Spec, header=header, cutoff_date=cutoff_date)
#     R = retransform_data(X=Y, Z=Z, Time=Time, Spec=Spec, header=header, cutoff_date=cutoff_date)

#     return Rhat, R, Time, cutoff_date

In [9]:
from estimation import select_model_by_r2, ml_fit_predict
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from lineartree import LinearForestRegressor, LinearBoostRegressor
from statsmodels.tsa.api import VAR

ref_date_col = "ReferenceDate"

def estimate_automl(
    ds, ds_base, spec, ref_date_col, series_name, reference_date, n_periods
):
    """
    Automatically trains models, evaluates them, and selects the best one based on R-squared.

    Parameters:
    - ds (pd.DataFrame): Dataset containing features and target.
    - ref_date_col (str): Column name for reference dates.
    - series_name (str): Column name for the series to forecast.
    - reference_date (datetime or str): Date for forecasting and backcasting split.
    - n_periods (int): Number of periods to forecast.

    Returns:
    - dict: A dictionary containing the best model, its R-squared score, and predictions.
    """
    models = {
        "LinearRegression": LinearRegression(),
        "Ridge": Ridge(),
        "LinearForest": LinearForestRegressor(
            base_estimator=Ridge(), random_state=42, max_features="log2"
        ),
        "LinearBoost": LinearBoostRegressor(
            base_estimator=Ridge(), random_state=42, max_features="log2"
        ),
        "RandomForestRegressor": RandomForestRegressor()
    }

    models_results = {}

    for model_name, model in models.items():
        coef_, pred, T, values = ml_fit_predict(
            ds=ds,
            ref_date_col=ref_date_col,
            model=model,
            series_name=series_name,
            reference_date=reference_date,
            n_periods=n_periods,
        )

        models_results[model_name] = {
            "backcast": pred["y_pred"].drop(reference_date),
            "forecast": pred["y_pred"].loc[reference_date],
            "reference_date": reference_date,
            "coef_": coef_,
            "values": values
        }
    # Ensure all predictions align with the actuals index
    y_actual = ds.set_index(ref_date_col).loc[T].sort_index()[series_name]

    # Select the best model based on R-squared
    best_model_res= select_model_by_r2(
        models_results, y_actual.drop(reference_date)
    )
    best_model_res["actual"] = y_actual
    best_model_res["rmse"] = rmse(
        actual=y_actual.drop(reference_date),
        predicted=models_results[model_name]["backcast"],
    )
    best_model_res["mape"] = mape(
        actual=y_actual.drop(reference_date),
        predicted=models_results[model_name]["backcast"],
    )

    # Rhat, R, Time, cutoff_date = cast_to_base_unit(
    #     ds=ds_base, model_result=best_model_info, spec=spec, series_name=series_name
    # )
    # TBC

    return best_model_res

In [10]:
def cast_to_base_unit(
        ds, 
        # model_result, 
        spec, 
        series_name,
        series_values,
        dtype
        ):
    Spec = cast_spec_to_dict(spec.loc[spec["seriesid"] == series_name])

    ## Retransform
    ds = _convert_to_datetime(ds, ["ReferenceDate"])

    dsrc = ds.set_index("ReferenceDate")

    # def retransform_prediction(transf_series, base_series, Spec, series_name):
    base_series = dsrc[series_name]
    header = [series_name]

    # backcast = model_result["pred_"]["backcast"]
    # forecast = pd.Series(
    #     model_result["pred_"]["forecast"],
    #     index=[model_result["pred_"]["reference_date"]],
    # )
    # transf_pred = pd.concat([backcast, forecast])
    # transf_pred.index = pd.to_datetime(transf_pred.index)

    # transf_series = model_result["actual"]

    Time = np.sort(
        np.unique(np.concatenate((base_series.index.date, series_values.index.date)))
    )
    cutoff_date = series_values.index.min().date()

    Z = base_series.reindex(Time).to_numpy().reshape(-1, 1)

    Y = series_values.reindex(Time).to_numpy().reshape(-1, 1)
    if dtype == "actual":
        R = retransform_data(
            X=Y, Z=Z, Time=Time, Spec=Spec, header=header, cutoff_date=cutoff_date
        )
    elif dtype == "pred":
        R = retransform_(
            X=Y, Z=Z, Time=Time, Spec=Spec, header=header, cutoff_date=cutoff_date
        )
    else:
        raise Exception(f"ValueError: {dtype} not supported")

    return R, Time, cutoff_date

In [11]:
#!/usr/bin/env python
# coding: utf-8

import numpy as np
import pandas as pd

from typing import List

import plotly.graph_objects as go


def convert_to_datetime(df: pd.DataFrame, colnames: List[str]) -> pd.DataFrame:
    for col in colnames:
        df[col] = pd.to_datetime(df[col])
    return df


def plot_prediction(
    dt: pd.Series,
    y_pred: pd.Series,
    y_actual: pd.Series,
    legend_position: tuple = (0,0),
    # y_id: str,
    title: str = "",
    mode="markers",
    tickfont_size=14,
    lower1: pd.Series = None,
    upper1: pd.Series = None,
    lower2: pd.Series = None,
    upper2: pd.Series = None,
    plt_out_path: str = None
    ) -> None:

    body = []
    if np.all([lower1, lower2, upper1, upper2]):
        body = [
            go.Scatter(
                name="Estimate + 3 Standard Deviations",
                x=dt,
                y=upper2,
                mode="lines",
                marker=dict(color="#E7E8F0"),
                line=dict(width=0),
                showlegend=False,
            ),
            go.Scatter(
                name="Estimate - 3 Standard Deviations",
                x=dt,
                y=lower2,
                marker=dict(color="#E7E8F0"),
                line=dict(width=0),
                mode="lines",
                fillcolor="#E7E8F0",
                fill="tonexty",
                showlegend=False,
            ),
            go.Scatter(
                name="Estimate + Standard Deviation",
                x=dt,
                y=upper1,
                mode="lines",
                marker=dict(color="#BDC1D6"),
                line=dict(width=0),
                showlegend=False,
            ),
            go.Scatter(
                name="Estimate - Standard Deviation",
                x=dt,
                y=lower1,
                marker=dict(color="#BDC1D6"),
                line=dict(width=0),
                mode="lines",
                fillcolor="#BDC1D6",
                fill="tonexty",
                showlegend=False,
            ),
            ]

    body = body + [
        go.Scatter(
            name="Forecast",
            x=dt,
            y=y_pred,
            mode=mode,
            # line=dict(color='rgb(225, 69, 0)', width=2),  # rgb(216, 129, 71)
            line=dict(color="#841E62", width=1.5),
            marker={"size": 6},
            opacity=0.85,
        ),
        go.Scatter(
            name="Actual",
            x=dt,
            y=y_actual,
            mode=mode,
            marker={"size": 6, "symbol": "diamond"},
            line=dict(color="#000000", width=1.5),  # #7BCC62 / #68b562 / #7BB562
            opacity=0.85,
        )
    ]

    fig = go.Figure(body)

    fig.update_layout(
        autosize=False,
        width=900,
        height=650,
        plot_bgcolor="white",
        yaxis_title="",
        title=title,
        hovermode="x",
        legend=dict(
            bgcolor='rgba(0, 0, 0, 0)',  # Set background to transparent
            bordercolor='rgba(0, 0, 0, 0)',  # Optional: remove border
            orientation="h",
            yanchor="auto",
            x=legend_position[0],
            y=legend_position[1],
            xanchor="right",  # changed
            indentation=15,  # Increase the spacing between legend items
            font=dict(size=tickfont_size),  
            ),
    )

    fig.update_xaxes(
        mirror=True,
        ticks="outside",
        showline=True,
        linecolor="black",
        gridcolor="lightgrey",
        tickfont_size=tickfont_size,
    )
    fig.update_yaxes(
        mirror=True,
        ticks="outside",
        showline=True,
        linecolor="black",
        gridcolor="lightgrey",
        tickfont_size=tickfont_size,
    )

    if plt_out_path:
        fig.write_image(plt_out_path)

    fig.show()
    # return fig


In [12]:
series_name = "PCEC96"
reference_date = "2024-08-01"
n_periods = 60
plt_out_dir = "../data/07_model_output/"

In [13]:
ds = pd.read_parquet("../data/04_feature/selected_series.parquet")

In [14]:
ds_spec = pd.read_csv("../data/02_intermediate/variable.csv")
ds_base = pd.read_parquet("../data/02_intermediate/non_transformed_data.parquet")

# Example usage
best_model_result = estimate_automl(
    ds=ds,
    ds_base=ds_base,
    spec=ds_spec,
    ref_date_col="ReferenceDate",
    series_name=series_name,
    reference_date=reference_date,
    n_periods=n_periods,
)

# Print the best model's details
print("===== Best Model Details =====")
print(f"Model                     : {best_model_result['best_model']}")
print(f"Reference Date            : {reference_date}")
print(f"Forecast                  : {best_model_result['pred_']['forecast']:.4f}")
print(f"R-Squared (R²)            : {best_model_result['r_squared']:.4f}")
print(f"Mean Absolute Percentage Error (MAPE): {best_model_result['mape']:.2f}%")
print(f"Root Mean Square Error (RMSE) : {best_model_result['rmse']:.4f}")

formula = ds_spec.loc[ds_spec["seriesid"] == series_name]["transformation"].item()
unit = ds_spec.loc[ds_spec["seriesid"] == series_name]["units"].item()

dt = best_model_result['pred_']['backcast'].index
pred1 = best_model_result['pred_']['backcast']
actual1 = best_model_result['actual'].loc[dt]
bounds = calculate_conf_bounds(pred1, actual1)

plot_prediction(
    dt=dt,
    y_pred=pred1,
    y_actual=actual1,
    mode="lines+markers",
    title=f"Series: {series_name}, Reference Date: {reference_date}, Unit: {unit} {formula}",
    )
plot_prediction(
    dt=dt,
    y_pred=pred1,
    y_actual=actual1,
    mode="lines+markers",
    lower1=bounds["L1"],
    upper1=bounds["U1"],
    lower2=bounds["L2"],
    upper2=bounds["U2"],
    title=f"Series: {series_name}, Reference Date: {reference_date}, Unit: {unit}",
    )

transf_pred = pd.concat([
    best_model_result["pred_"]["backcast"],
    pd.Series(
        best_model_result["pred_"]["forecast"],
        index=[pd.to_datetime(best_model_result["pred_"]["reference_date"])],
    )
    ])
transf_actual = best_model_result["actual"]


Rhat, Time, cutoff_date = cast_to_base_unit(ds_base, ds_spec, series_name, transf_pred, dtype="pred")
R, _, _ = cast_to_base_unit(ds_base, ds_spec, series_name, transf_actual, dtype="actual")

# TBC
header = [series_name]
Rhat_df = pd.DataFrame(Rhat, columns=header, index=Time)
R_df = pd.DataFrame(R, columns=header, index=Time)

reference_date = pd.to_datetime(reference_date).date()
retr_forecast = Rhat_df.loc[reference_date].item()
retr_actual = R_df.loc[reference_date].item()

print("\n===== Forecast vs Actual =====")
print(f"Reference Date            : {reference_date}")
print(f"Forecast                  : {best_model_result['pred_']['forecast']:.4f}")
print(f"Forecast (retransformed)  : {retr_forecast:,.2f}")
print(f"Actual Release            : {retr_actual:,.2f}")
print(f"Percentage Error (Level)  : {(retr_forecast - retr_actual) / retr_actual:.2%}")

Z_df = pd.DataFrame(ds[series_name], columns=header, index=Time)

pred2 = Rhat_df.loc[dt][series_name]
actual2 = R_df.loc[dt][series_name]

plot_prediction(
    dt=dt,
    y_pred=pred2,
    y_actual=actual2,
    mode="lines+markers",
    title=f"Series: {series_name}, Reference Date: {reference_date}, Unit: {unit}",
    )

bounds_level = {}
for key, value in bounds.items():
    data, dt_, _ = cast_to_base_unit(ds_base, ds_spec, series_name, value, dtype="pred")
    tmp = pd.Series(data.reshape(1, -1)[0], index=dt_)
    bounds_level[key] = tmp.loc[dt]


plot_prediction(
    dt=dt,
    y_pred=pred2,
    y_actual=actual2,
    lower1=bounds_level["L1"],
    upper1=bounds_level["U1"],
    lower2=bounds_level["L2"],
    upper2=bounds_level["U2"],
    mode="lines+markers",
    title=f"Series: {series_name}, Reference Date: {reference_date}, Unit: {unit}",
    )

===== Best Model Details =====
Model                     : Ridge
Reference Date            : 2024-08-01
Forecast                  : 0.1128
R-Squared (R²)            : 0.9012
Mean Absolute Percentage Error (MAPE): 4.60%
Root Mean Square Error (RMSE) : 1.7972



===== Forecast vs Actual =====
Reference Date            : 2024-08-01
Forecast                  : 0.1128
Forecast (retransformed)  : 15,888.21
Actual Release            : 16,089.70
Percentage Error (Level)  : -1.25%
